In [0]:
%sql
LIST '/Volumes/bazar9_dbw/default/bronze';

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bazar9_dbw.bronze;
CREATE SCHEMA IF NOT EXISTS bazar9_dbw.silver;
CREATE SCHEMA IF NOT EXISTS bazar9_dbw.gold;

In [0]:
from pyspark.sql import functions as F

CATALOG = "bazar9_dbw"
SOURCE_PATH = "/Volumes/bazar9_dbw/default/bronze"

files = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "product_category_translation",
}

for file_name, table_name in files.items():
    df = (
        spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(f"{SOURCE_PATH}/{file_name}")
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )
    (
        df.write
        .mode("overwrite")
        .option("overwriteSchema", True)
        .saveAsTable(f"{CATALOG}.bronze.{table_name}")
    )
    print(f"{table_name}: {df.count():,} rijen")
    